# Context Engineering- Budget, Order, and Retrieve the Working Memory

**Module 5 · Lesson 5.4**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- Context is working memory - the token budget
- Context rot - why more context can hurt
- Order and structure - put signal where it lands
- Retrieve, don't stuff - the hybrid default
- Manage growing context - trim, compress, summarize
- Cache and agentic context - compaction and memory

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install dependencies (if running in Colab or fresh environment)
# Uncomment the next line if needed:
# !pip install numpy -q

# Reproducibility - the lesson uses seeded random values throughout
import numpy as np
np.random.seed(42)

## The whole game: the answer was in the prompt, and the model still missed it

> 🧮 **Analogy**
>
> **Your context window is a desk, not a warehouse.** Only so much fits on it, and you read the top and bottom sheets most clearly. Pile every file onto it "just in case" and you bury the one page that mattered - and you slow yourself down reading the rest.
> A model treats its context the same way. Everything you send - system prompt, examples, retrieved documents, conversation history, the user's question - shares one finite, attention-limited window. **Context engineering is the discipline of deciding what goes on the desk, and where.** Reliability comes from the context system, not from clever wording.
> **Where the analogy breaks down:** unlike you at a desk, the model cannot choose to look harder at the middle - its attention is structurally weakest there (lost-in-the-middle). And irrelevant pages do not sit quietly; semantically similar but off-topic text actively distracts it.

Every technique here is tested with **real API calls** and paired with a plain-English walkthrough. We use Gemini through the current unified SDK, but the discipline transfers to Claude and GPT.

- **Explain** why long context degrades - lost-in-the-middle and context rot - and predict where a buried fact will be missed.

- **Construct** a token budget for a request and order the context so the highest-signal tokens sit where the model attends best.

- **Apply** the 2026 hybrid default - retrieve the relevant tokens, then reason over them - instead of stuffing the whole corpus.

- **Evaluate** context-management tactics (trimming, compression, summarization, caching, compaction) and choose by cost, latency, and reliability.

## Warm-up: 60 seconds of recall

Tap each card to check yourself against earlier lessons. Each one is load-bearing for today.

## The setup: helpers we will reuse all lesson

Every example calls `ask()` (a completion) or `count_tokens()` (measure the budget), both on the current unified **google-genai** SDK (the older `google.generativeai` package was deprecated in 2025 - [migration guide](https://ai.google.dev/gemini-api/docs/migrate)).

**📝 `setup.py Gemini`** - *API*

In [ ]:
# pip install "google-genai>=1.0.0"
from google import genai
from google.genai import types
import os

client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
MODEL = "gemini-3-flash"

def ask(prompt: str, temperature: float = 0.2, system: str = None) -> str:
    cfg = types.GenerateContentConfig(temperature=temperature, system_instruction=system)
    try:
        return client.models.generate_content(model=MODEL, contents=prompt, config=cfg).text.strip()
    except Exception as e:
        return f"[API error: {e}]"   # teaching-only: real code raises/retries - never let an error string become a summary or a cached prefix

def count_tokens(text: str) -> int:
    """Measure tokens BEFORE sending - you can't manage what you don't measure."""
    return client.models.count_tokens(model=MODEL, contents=text).total_tokens

print(count_tokens("How many tokens is this sentence?"))
# Output: 8

- `genai.Client(...)` - one reusable client (the deprecated package used a global `configure()` plus per-call `GenerativeModel`).

- `count_tokens()` calls the real `client.models.count_tokens` endpoint - the only honest way to know how much of the budget a piece of text uses before you send it.

- Tokens, not characters or words: an Indic-script document can be 2-3x the tokens of its English equivalent (the Lesson 4.1 token tax), so always measure the real thing.

- The `try/except` keeps one network blip from crashing a pipeline that may make several context-management calls.

---

## 🎯 Concept 1: Context is working memory - the token budget

### Context is working memory - the token budget

Every token you send competes for one finite window. Budget it on purpose.

**A fixed wallet.** You have a set amount to spend across rent, food, and savings. Spend it all on one thing and the others get nothing - and you cannot spend money you do not have.

The gain: the context window is that wallet, paid in tokens. Every component is a line item, and the answer needs its own reserved budget.

**📝 `token_budget.py Gemini`** - *API*

In [ ]:
def fits_budget(parts: dict, window: int = 200_000, reserve_out: int = 2_000) -> dict:
    """Sum the token cost of every context part; check it leaves room for the answer."""
    used = {name: count_tokens(text) for name, text in parts.items()}
    total = sum(used.values())
    free = window - total - reserve_out
    return {"per_part": used, "total": total, "free_for_answer": free, "fits": free >= 0}

ctx = {
    "system": "You are a support assistant...",        # small, fixed
    "examples": open("few_shot.txt").read(),           # medium, fixed
    "history": open("chat_history.txt").read(),        # grows every turn
    "retrieved": open("top_docs.txt").read(),         # varies per query
}
print(fits_budget(ctx))
# Output: {'per_part': {'system': 6, 'examples': 7900, 'history': 31200,
# Output:  'retrieved': 162000}, 'total': 201106, 'free_for_answer': -3106, 'fits': False}

- **It counts every part separately** with `count_tokens`, so you can see which component is the budget hog (here: the 162k of retrieved docs).

- **It reserves output space up front** (`reserve_out`) - a window that is full to the brim leaves nothing for the model to answer with.

- **`free_for_answer` goes negative** when you overflow, and `fits` turns False - the signal to trim history or retrieve less *before* you send.

- This is the "measure before you manage" habit: the rest of the lesson is what you do once the budget says it does not fit.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

#### 💡 What just happened

⚡ What just happened?The context window is a zero-sum token budget shared by system prompt, examples, history, retrieved docs, and the answer. You cannot manage what you do not measure, so `count_tokens` comes first; and you must reserve output space before filling the rest. **Everything that follows is how to spend this budget well.**

---

## 🎯 Concept 2: Context rot - why more context can hurt

### Context rot - why more context can hurt

The counterintuitive failure mode: accuracy falls as the window fills.

> 📦 **Info**
>
> Two compounding effects
> - **Lost-in-the-middle (Liu et al. 2023):** accuracy is U-shaped in the position of the key fact - strong at the edges, weak in the middle, a 30%+ drop when buried.
> - **Context rot (Chroma 2025):** across 18 frontier models, accuracy *fell* as input length grew - some holding ~95% then sliding toward ~60% past a threshold - through attention dilution (the model has a fixed budget of "attention" to spread across all the tokens, so more tokens means each one gets a thinner slice) and distractor interference. A counterintuitive twist: coherent, well-structured filler degraded attention *more* than shuffled filler.

> 💡 **Info**
>
> ⚠️ Common mistake"The window is 1M tokens, so I can just paste everything in." A bigger window does **not** repeal lost-in-the-middle or context rot - effective context still trails advertised context (the RULER benchmark). More room to stuff is not permission to stuff.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

#### 💡 What just happened

⚡ What just happened?Long context is real but lossy: a fact's *position* determines whether the model uses it, and piling on more tokens dilutes attention and invites distractors. **"If it is in the context, the model will use it" is false.** That single fact is why the rest of the lesson - order, retrieve, compress - exists.

---

## 🎯 Concept 3: Order and structure - put signal where it lands

### Order and structure - put signal where it lands

Since position matters, place the highest-signal content where the model attends.

**A good briefing.** You lead with the one thing the person must know and end with the action you want - you do not bury the ask on page 7.

The gain: order the context like a briefing. Critical instruction at the edges; bulk material in the middle; clear section headers between them.

**📝 `order_context.py`** - *Pattern*

In [ ]:
def build_context(task: str, docs: list[str], question: str) -> str:
    """Signal at the edges, bulk reference in the middle, fenced sections."""
    docs_block = "\n\n".join(f"<doc id={i}>\n{d}\n</doc>" for i, d in enumerate(docs))
    return f"""# TASK (read this first)
{task}

# REFERENCE MATERIAL
{docs_block}

# QUESTION (answer this, using only the reference above)
{question}"""     # the ask sits LAST - recency works for you

prompt = build_context(
    task="Answer strictly from the reference. If it is not there, say 'not found'.",
    docs=["Refund policy: defective items may be returned within 7 days of delivery for a full refund.",
          "...policy B...", "...policy C..."],
    question="What is the refund window for a defective item?")
print(ask(prompt))
# Output: 7 days from delivery for a defective item (per doc 1). [grounded, on-task]

- **The task leads.** The first thing the model reads is the instruction and the grounding rule ("only from the reference"), which sits in a high-attention position.

- **Bulk docs go in the middle** - the lowest-attention zone - which is the right place for material the model only needs to consult, not obey.

- **The question sits last,** exploiting recency so the actual ask is freshest when the model starts generating.

- **Fenced `<doc>` sections** (the delimiter instinct from Lesson 5.1) let the model tell documents apart and cite by id, and make it harder for one doc's text to be mistaken for an instruction.

#### 💡 What just happened

⚡ What just happened?Because attention is position-dependent, ordering is an engineering lever: lead with the instruction, end with the ask, and put bulk reference where rot hurts least, all fenced into labelled sections. **Same tokens, better placement, more reliable answer.**

---

## 🎯 Concept 4: Retrieve, don't stuff - the hybrid default

### Retrieve, don't stuff - the hybrid default

Send the relevant few tokens, then reason over them - not the whole corpus.

> 🔍 **Analogy**
>
> **Studying the right pages.** For one exam question you do not re-read the whole textbook - you find the few relevant pages and study those. Retrieval does the same: embed the query (turn the text into a vector of numbers whose position captures its meaning), find the documents whose vectors sit nearest, and send only those (the embeddings idea from Lesson 4.1).
> **Where it breaks down:** retrieval is only as good as its precision. Pull in semantically similar but wrong passages and you have added distractors - sometimes worse than sending nothing, because near-misses actively mislead.

**📝 `retrieve_then_reason.py Gemini`** - *API*

In [ ]:
import numpy as np

def embed(text: str) -> np.ndarray:
    r = client.models.embed_content(model="gemini-embedding-001", contents=text)
    return np.array(r.embeddings[0].values)

def retrieve(query: str, corpus: list[str], k: int = 3) -> list[str]:
    """Return the k documents nearest the query. DEMO ONLY: this re-embeds every
    doc on each call. In production, embed the corpus ONCE into a vector DB
    (Module 8) - never embed on the request path."""
    q = embed(query)
    qn = np.linalg.norm(q)
    scored = []
    for d in corpus:
        e = embed(d)                                   # embed each doc once, not twice
        scored.append((float(np.dot(q, e) / (qn * np.linalg.norm(e))), d))
    scored.sort(reverse=True)
    return [d for _, d in scored[:k]]

question = "What is the refund window for a defective item?"
top = retrieve(question, big_corpus, k=3)         # 3 of 500, not all 500
answer = ask(build_context("Answer only from the reference.", top, question))
print(answer)
# Output: 7 days from delivery for a defective item. [from the 3 retrieved docs]

- **Embed and rank.** `retrieve` embeds the query and every doc, then scores by *cosine similarity* - how closely two embedding vectors point in the same direction (1.0 = near-identical meaning, ~0 = unrelated) - and keeps the top `k` (the KATE idea from Lesson 5.2).

- **Then reason over the subset.** The retrieved docs feed the same ordered `build_context` from Step 3 - retrieval and long-context are partners, not rivals.

- **Why it beats stuffing:** the model sees 3 high-signal docs instead of 500, so there is no middle to get lost in and no distractor load - higher accuracy at a fraction of the tokens.

- In production you embed the corpus once and store the vectors (a vector database) - the full retrieval stack is Module 8 (RAG). Here it is the decision: retrieve, do not stuff.

- **Retrieved content is untrusted.** These docs come from outside your prompt, so a poisoned document can carry an *indirect prompt injection* (OWASP LLM01:2025) - text that tries to hijack the model. Keep it fenced (Step 3) and never let retrieved text become instructions; the defenses are Lesson 5.7.

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

A team with a 1M-token model drops a 300-page manual into every request "so the model has everything." Accuracy *drops* (the answer is lost in the middle among hundreds of irrelevant pages), cost balloons, and latency triples. The fix is retrieval: find the handful of relevant sections and send only those. **A bigger window did not make stuffing a good idea - it just made it possible.**

#### 💡 What just happened

⚡ What just happened?The 2026 default is hybrid: retrieve the relevant tokens (roughly 50K-200K, not a whole corpus), then long-context-reason over them. Retrieval beats stuffing on accuracy, cost, and latency at once, because a clean small context has no middle to get lost in. **Long context did not make retrieval obsolete - it made the hybrid the default.**

---

## 🎯 Concept 5: Manage growing context - trim, compress, summarize

### Manage growing context - trim, compress, summarize

Conversations and documents outgrow the window. Shrink them without losing the thread.

**📝 `trim_and_summarize.py`** - *Production*

In [ ]:
def compact_history(system: str, turns: list[str], keep_recent: int = 6) -> str:
    """Keep system + the last N turns verbatim; summarize the older middle."""
    if len(turns) <= keep_recent:
        return system + "\n" + "\n".join(turns)
    old, recent = turns[:-keep_recent], turns[-keep_recent:]
    summary = ask("Summarize these earlier turns into 5 bullet facts to remember "
                  "(ids, names, decisions). Keep it under 120 words:\n" + "\n".join(old))
    return f"{system}\n\n# EARLIER (summary)\n{summary}\n\n# RECENT\n" + "\n".join(recent)

ctx = compact_history("You are a support agent.", all_turns, keep_recent=6)
print(count_tokens(ctx))
# Output: 1840          (was 31200 raw - a ~17x reduction, thread preserved)
# Note: summarizing is not free - it adds a model round-trip (latency + tokens) and another
# point that can fail, so cache the summary and recompact periodically, not every turn.

- **Keep the privileged parts verbatim.** The system prompt and the most recent `keep_recent` turns stay exact - those sit in high-attention positions and carry the live thread.

- **Summarize the old middle, do not delete it.** The older turns are compressed into a few bullet facts (ids, names, decisions) so nothing load-bearing is silently dropped.

- **Massive token reduction.** 31k of raw history becomes ~1.8k - the budget from Step 1 is recovered without losing the conversation's memory.

- For documents larger than the window, the same idea scales up as a *map-reduce summarization chain*: summarize each chunk, then summarize the summaries.

#### 💡 What just happened

⚡ What just happened?Growing context is managed by trimming and compression: keep the system prompt and recent turns verbatim, summarize the old middle, and map-reduce documents too big to fit. **You trade a little fidelity on old turns for a large budget recovery** - and you keep the answer space the budget (Step 1) demands.

---

## 🎯 Concept 6: Cache and agentic context - compaction and memory

### Cache and agentic context - compaction and memory

Make the stable context cheap, and hand long-running sessions to automatic management.

> 📦 **Info**
>
> Cache the stable, vary the fresh
> Structure every request as a **cached prefix** (system prompt, policies, fixed few-shot examples, long stable documents) plus a **fresh suffix** (this turn's retrieved docs and the user's question). The prefix is cached and billed at roughly a tenth on reuse, so a large stable context costs almost nothing after the first call - the same prefix/suffix structure from Lesson 5.2, now applied to whole documents.

> ✅ **Info**
>
> Agentic context management (2026)
> When a long-running agent's context approaches the window limit, the 2026 tooling manages it automatically:
> - **Compaction** - summarize older context into a structured snapshot (goal, key facts, file state, plan). Claude Code auto-compacts near ~80% of the window; Gemini CLI compresses nearer ~50%.
> - **Tool-result clearing** - drop large tool outputs once they have been used, instead of carrying them forever.
> - **Note-taking memory** - the agent writes durable notes *outside* the window and pulls them back when needed.
> - **Subagents** - isolate a subtask in its own fresh context so it does not pollute the main window.
> These are the foundations of agent memory, built out fully in Module 11.

**📝 `context_pipeline.py Complete`** - *project*

In [ ]:
def answer_with_context_engineering(question: str, corpus: list[str],
                                   history: list[str], system: str) -> str:
    """The whole lesson in one pipeline: retrieve -> compact -> order -> budget -> answer."""
    docs = retrieve(question, corpus, k=3)                  # Step 4: don't stuff
    convo = compact_history(system, history, keep_recent=6)     # Step 5: compress old turns
    prompt = build_context(convo, docs, question)               # Step 3: order it
    budget = fits_budget({"prompt": prompt})                    # Step 1: check it fits
    if not budget["fits"]:
        docs = retrieve(question, corpus, k=2)              # over budget? retrieve less
        prompt = build_context(convo, docs, question)
    return ask(prompt)

print(answer_with_context_engineering(q, big_corpus, chat_history, system_prompt))
# Output: a grounded, on-task answer from a context that fits and stays high-signal

- **Retrieve (Step 4)** picks the few relevant docs, so the context starts high-signal.

- **Compact (Step 5)** shrinks the chat history without losing the thread.

- **Order (Step 3)** places task, docs, and question where the model attends best.

- **Budget (Step 1)** checks it fits and reserves answer space - and if it overflows, it retrieves *less* rather than truncating blindly.

- In production the stable system prompt would be a cached prefix, and a long-running agent would hand history to compaction (Module 11). This is context engineering as a system, not a single prompt.

#### 💡 What just happened

⚡ What just happened?Caching makes a large stable context nearly free, and agentic management (compaction, tool-result clearing, memory, subagents) keeps a long session's window high-signal automatically. The pipeline shows the whole discipline in order: retrieve, compact, order, budget. **Context engineering is a system you design, not a prompt you write.**

## Putting it together: the context discipline

### 🧮 The whole lesson on one screen

**Context is finite working memory** - a token budget shared by every part (Step 1). **More context can hurt** - lost-in-the-middle and context rot mean a buried fact is a missed fact (Step 2). So **order it** - signal at the edges, bulk in the middle, fenced sections (Step 3); **retrieve, do not stuff** - the hybrid default sends the relevant few (Step 4); **compress what grows** - trim and summarize history (Step 5); and **cache the stable, compact the rest** - make it cheap and hand long sessions to automatic management (Step 6). Above all: the smallest high-signal context wins.

Forward hooks you just planted: structured output comes next in Lesson 5.5; we'll build the full retrieval stack in Module 8 (RAG); agent memory and compaction grow up in Module 11; and we'll come back to context cost in Module 13.

- Liu et al., *Lost in the Middle: How Language Models Use Long Contexts* (2023) - [arxiv.org/abs/2307.03172](https://arxiv.org/abs/2307.03172)

- Hong et al. (Chroma), *Context Rot: How Increasing Input Tokens Impacts LLM Performance* (2025) - [research.trychroma.com/context-rot](https://research.trychroma.com/context-rot)

- Xu et al., *Retrieval meets Long Context Large Language Models* (2023) - [arxiv.org/abs/2310.03025](https://arxiv.org/abs/2310.03025)

- Hsieh et al., *RULER: What's the Real Context Size of Your Long-Context LLM?* (2024) - [arxiv.org/abs/2404.06654](https://arxiv.org/abs/2404.06654)

- Lewis et al., *Retrieval-Augmented Generation* (2020) - [arxiv.org/abs/2005.11401](https://arxiv.org/abs/2005.11401)

- Anthropic, *Effective context engineering for AI agents* (2025) - [anthropic.com/engineering](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents); [compaction docs](https://platform.claude.com/docs/en/build-with-claude/compaction)

- Google Gemini [long-context guide](https://ai.google.dev/gemini-api/docs/long-context) and [google-genai migration guide](https://ai.google.dev/gemini-api/docs/migrate)

## Key takeaways

> ℹ️ **Info**
>
> What you learned - 6 ideas
> - **Context is a token budget** - measure with `count_tokens` and reserve output space before filling the window.
> - **Context rot** - lost-in-the-middle and attention dilution mean more context can lower accuracy; a bigger window does not fix it.
> - **Order and structure** - signal at the edges, bulk in the middle, fenced labelled sections; the ask goes last.
> - **Retrieve, don't stuff** - the hybrid default sends the relevant few tokens, winning on accuracy, cost, and latency.
> - **Manage growth** - trim and summarize history; map-reduce documents bigger than the window.
> - **Cache and compact** - cache the stable prefix; hand long sessions to compaction, memory, and subagents (Module 11).

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **Context Engineering- Budget, Order, and Retrieve the Working Memory**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-5_4.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-5.4.html` - regenerate this notebook whenever the source HTML is updated.*
